In [ ]:
# SCRIPT DE SIMULACION A LA BASE DE DATOS DE TEST (versión ajustada)
# SIMULADOR DE DATOS - ALINEADO CON UMBRALES DE ALERTAS (EcoPulse)
import time
import random
import psycopg2
import numpy as np
from datetime import datetime

# --- Últimos valores para suavizar transiciones ---
last_values = {
    "tem": None, "hum": None, "pres": None,
    "mp1_0_stp": None, "mp2_5_stp": None, "mp10_stp": None,
    "mp1_0_ate": None, "mp2_5_ate": None, "mp10_ate": None,
    "co2": None, "dir_viento": None, "rap_viento": None,
    "consumo_1": None, "consumo_2": None, "consumo_3": None,
    "agua_caida": None
}

def suavizar(nombre, esperado, variacion=1.0):
    if last_values[nombre] is None:
        val = esperado + random.uniform(-variacion, variacion)
    else:
        val = last_values[nombre] + random.uniform(-variacion, variacion)
        val = (val + esperado) / 2
    last_values[nombre] = val
    return val

def transformar_valores(valores):
    """Convierte None o NaN en -999"""
    return [-999 if (v is None or (isinstance(v, float) and np.isnan(v))) else v for v in valores]

def generar_datos():
    ahora = datetime.now()
    hora = ahora.hour
    mes = ahora.month
    dia_semana = ahora.weekday()

    # --- Temperatura y humedad de referencia ---
    if mes in [12,1,2]:  t_min, t_max = 14, 32
    elif mes in [6,7,8]: t_min, t_max = 5, 18
    else:                 t_min, t_max = 10, 25

    frac = hora/15 if hora < 15 else (hora-15)/9
    if hora < 15:
        temp_esp = t_min + (t_max - t_min) * (frac ** 1.5)
    else:
        temp_esp = t_max - (t_max - t_min) * (frac ** 0.7)

    tem_bme280 = round(suavizar("tem", temp_esp, 0.3), 1)
    hum_bme280 = max(5, round(suavizar("hum", 90 - (tem_bme280 - t_min)*2, 1), 1))
    pres_bme280 = round(suavizar("pres", 1010, 0.1) + random.uniform(-1,1), 1)
    alt_bme280 = 570.0

    # --- Material particulado base ---
    factor_trafico = 1 if (7 <= hora <= 9 or 18 <= hora <= 21) and dia_semana < 5 else 0
    factor_invierno = 1.5 if mes in [5,6,7,8] else 1.0
    mp25_esp = 15 + 20 * factor_trafico * factor_invierno

    mp2_5_stp = round(suavizar("mp2_5_stp", mp25_esp, 1.5), 1)
    mp10_stp  = round(suavizar("mp10_stp", mp2_5_stp*1.2, 2), 1)
    mp1_0_stp = round(suavizar("mp1_0_stp", mp2_5_stp*0.7, 1), 1)

    mp1_0_ate = round(suavizar("mp1_0_ate", mp1_0_stp, 2), 1)
    mp2_5_ate = round(suavizar("mp2_5_ate", mp2_5_stp, 3), 1)
    mp10_ate  = round(suavizar("mp10_ate", mp10_stp, 3), 1)

    mp_gt_03 = mp2_5_stp * random.uniform(50,200)
    mp_gt_05 = mp2_5_stp * random.uniform(20,100)
    mp_gt_1  = mp2_5_stp * random.uniform(2,20)
    mp_gt_25 = mp2_5_stp * random.uniform(0.5,5)
    mp_gt_5  = mp2_5_stp * random.uniform(0.2,2)
    mp_gt_10 = mp2_5_stp * random.uniform(0.1,1)

    # --- CO₂ base ---
    co2_esp = 420 + mp2_5_stp*2 + (200 if mes in [6,7,8] else 0)
    co2_mhz19 = round(suavizar("co2", co2_esp, 5), 1)

    # --- Viento y agua ---
    dir_viento = round(suavizar("dir_viento", random.uniform(0,360), 5), 1)
    rap_viento = round(suavizar("rap_viento", 2 if hora<12 else 5, 0.5), 1)
    agua_caida = round(suavizar("agua_caida", random.uniform(0,0.5), 0.05), 2)

    consumo_1 = round(suavizar("consumo_1", 50, 5), 1)
    consumo_2 = round(suavizar("consumo_2", 20, 3), 1)
    consumo_3 = round(suavizar("consumo_3", 10, 2), 1)

    prob = random.random()

    # 🔧 Falla de sensores (10%)
    if prob < 0.10:
        valores = [
            tem_bme280, hum_bme280, pres_bme280, alt_bme280,
            mp1_0_stp, mp2_5_stp, mp10_stp,
            mp1_0_ate, mp2_5_ate, mp10_ate,
            mp_gt_03, mp_gt_05, mp_gt_1, mp_gt_25, mp_gt_5, mp_gt_10,
            co2_mhz19, dir_viento, rap_viento, agua_caida,
            consumo_1, consumo_2, consumo_3
        ]
        fallas = random.sample(range(len(valores)), random.randint(1,2))
        for i in fallas: valores[i] = None
        print(f"⚠️ Error de sensor ({len(fallas)}) -> {fallas}")
        return transformar_valores(valores + [1])

    # 🌪️ Eventos extremos (15%)
    if prob < 0.25:
        evento = random.choice([
            "ola_calor", "ola_frio", "smog_extremo",
            "tormenta_viento", "tormenta_electrica", "lluvia_intensa"
        ])
        print(f"🌩️ Evento extremo generado: {evento}")

        if evento == "ola_calor":
            tem_bme280 = random.uniform(38, 46)      # ≥ 37 → alerta crítica
            hum_bme280 = random.uniform(5, 15)
            mp2_5_stp = random.uniform(70, 160)      # sobre PM pre-emergencia
            co2_mhz19 += random.uniform(200, 400)
            consumo_1 += random.uniform(30, 60)
        elif evento == "ola_frio":
            tem_bme280 = random.uniform(-8, -5)      # ≤ -5 → crítica
            hum_bme280 = random.uniform(85, 100)
            consumo_1 += random.uniform(40, 80)
            co2_mhz19 += random.uniform(150, 300)
        elif evento == "smog_extremo":
            mp2_5_stp = random.uniform(150, 400)     # ≥ 150 → emergencia
            mp10_stp  = mp2_5_stp + random.uniform(50,100)
            co2_mhz19 = random.uniform(1000, 3000)
            hum_bme280 = random.uniform(60,90)
            tem_bme280 -= random.uniform(0,2)
        elif evento == "tormenta_viento":
            rap_viento = random.uniform(33, 50)      # ≥33 m/s → huracán
            dir_viento = random.uniform(0,360)
            agua_caida = random.uniform(0.1,0.3)
        elif evento == "tormenta_electrica":
            rap_viento = random.uniform(30, 45)
            agua_caida = random.uniform(0.6, 0.9)
            pres_bme280 -= random.uniform(5,10)
            co2_mhz19 += random.uniform(150,300)
        elif evento == "lluvia_intensa":
            agua_caida = random.uniform(0.8, 1.2)    # simulando lluvia > 80 mm/día acumulado
            hum_bme280 = 100
            tem_bme280 = random.uniform(8,15)
            consumo_1 -= random.uniform(10,25)

    else:
        print("✅ Registro normal.")

    valores = [
        tem_bme280, hum_bme280, pres_bme280, alt_bme280,
        mp1_0_stp, mp2_5_stp, mp10_stp,
        mp1_0_ate, mp2_5_ate, mp10_ate,
        mp_gt_03, mp_gt_05, mp_gt_1, mp_gt_25, mp_gt_5, mp_gt_10,
        co2_mhz19, dir_viento, rap_viento, agua_caida,
        consumo_1, consumo_2, consumo_3, 1
    ]
    return transformar_valores(valores)

# --- Conexión a Cloud SQL ---
conn = psycopg2.connect(
    dbname="EcoPulse_Test",
    user="postgres",
    password="Admin!123",
    host="34.170.87.160",
    port="5432"
)
cur = conn.cursor()

try:
    while True:
        valores = generar_datos()
        cur.execute("""
            INSERT INTO datos_dispositivo (
                tem_bme280, hum_bme280, pres_bme280, alt_bme280,
                "mp1.0_stp", "mp2.5_stp", mp10_stp,
                "mp1.0_ate", "mp2.5_ate", mp10_ate,
                "mp_gt_0.3um", "mp_gt_0.5um", "mp_gt_1.0um",
                "mp_gt_2.5um", "mp_gt_5.0um", mp_gt_10um,
                co2_mhz19, dir_viento, rap_viento, agua_caida,
                consumo_1, consumo_2, consumo_3, id_dispositivo
            )
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        """, valores)
        conn.commit()
        print(f"{datetime.now()} -> Insertado {valores}\n")
        time.sleep(60)

except KeyboardInterrupt:
    print("Simulación detenida manualmente.")
    cur.close()
    conn.close()